Muhammad Haziq bin Abdullah (SW01083756), Mohamad Naqib bin Mustapa (SW01083743)

In [1]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from gensim import corpora
from gensim.models import LdaModel, CoherenceModel

In [2]:
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [3]:
df = pd.read_csv("news_dataset.csv", usecols=["text"])

In [4]:
df = df.dropna(subset=["text"]).copy()

stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", " ", text)   # remove punctuation and numbers
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words and len(word) > 2]
    tokens = [stemmer.stem(word) for word in tokens]
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return tokens

df["processed_text"] = df["text"].apply(preprocess_text)

# Remove empty rows after preprocessing
df = df[df["processed_text"].map(len) > 0].copy()

In [5]:
dictionary = corpora.Dictionary(df["processed_text"])
dictionary.filter_extremes(no_below=5, no_above=0.5)

corpus = [dictionary.doc2bow(text) for text in df["processed_text"]]

In [6]:
num_topics = 4

lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=num_topics,
    random_state=42,
    passes=10,
    iterations=100,
    alpha="auto",
    per_word_topics=True
)

In [7]:
print("\n========== LDA Topics ==========\n")
for idx, topic in lda_model.print_topics(num_topics=num_topics, num_words=10):
    print(f"Topic {idx + 1}: {topic}\n")


========== LDA Topics ==========

Topic 1: 0.009*"would" + 0.007*"one" + 0.007*"key" + 0.007*"use" + 0.007*"get" + 0.006*"like" + 0.006*"encrypt" + 0.006*"think" + 0.005*"know" + 0.005*"time"

Topic 2: 0.010*"peopl" + 0.008*"one" + 0.007*"would" + 0.006*"say" + 0.006*"god" + 0.005*"think" + 0.004*"know" + 0.004*"believ" + 0.004*"like" + 0.004*"right"

Topic 3: 0.030*"max" + 0.013*"game" + 0.011*"team" + 0.007*"play" + 0.006*"edu" + 0.006*"player" + 0.005*"year" + 0.005*"season" + 0.005*"leagu" + 0.005*"win"

Topic 4: 0.016*"use" + 0.009*"file" + 0.007*"system" + 0.006*"window" + 0.006*"program" + 0.006*"edu" + 0.006*"mail" + 0.006*"key" + 0.005*"one" + 0.005*"anonym"



In [8]:
coherence_model = CoherenceModel(
    model=lda_model,
    texts=df["processed_text"],
    dictionary=dictionary,
    coherence="c_v"
)

coherence_score = coherence_model.get_coherence()
print("Coherence Score:", coherence_score)

Coherence Score: 0.5321732775437141


In [9]:
def get_dominant_topic(bow):
    topic_probs = lda_model.get_document_topics(bow)
    dominant_topic = max(topic_probs, key=lambda x: x[1])[0]
    return dominant_topic

df["dominant_topic"] = [get_dominant_topic(doc) for doc in corpus]

print("\n========== Sample Topic Assignments ==========\n")
print(df[["text", "dominant_topic"]].head())


========== Sample Topic Assignments ==========

                                                text  dominant_topic
0  I was wondering if anyone out there could enli...               0
1  I recently posted an article asking what kind ...               0
2  \nIt depends on your priorities.  A lot of peo...               0
3  an excellent automatic can be found in the sub...               0
4  : Ford and his automobile.  I need information...               0


In [11]:
interpretation = f"""
Interpretation:
The coherence score of the LDA model is {coherence_score:.4f}. This score shows how well the
words in each topic fit together. A higher coherence score means the topics are clearer and
easier to understand. A lower score means some topics may contain mixed or less related words.
Overall, this score helps evaluate whether the 4 topics found by the model are meaningful.
"""
print(interpretation)


Interpretation:
The coherence score of the LDA model is 0.5322. This score shows how well the
words in each topic fit together. A higher coherence score means the topics are clearer and
easier to understand. A lower score means some topics may contain mixed or less related words.
Overall, this score helps evaluate whether the 4 topics found by the model are meaningful.

